In [1]:
import pandas as pd
import requests
from datetime import datetime
import time

/Library/Python/3.9/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
OUTPUT_PATH = "../data/snapshots_metadata.csv"
YEARS = range(2016, 2025)
companies = pd.read_csv("../data/companies.csv")
print(companies.shape)
companies.head()

(50, 4)


,ticker,company_name,sector,about_url
0,MSFT,Microsoft,Technology,https://www.microsoft.com/en-us/about
1,AAPL,Apple,Technology,https://investor.apple.com/our_values/default....
2,NVDA,NVIDIA,Technology,https://www.nvidia.com/en-us/about-nvidia/
3,GOOGL,Alphabet,Technology,https://about.google/company-info/
4,META,Meta Platforms,Technology,https://www.meta.com/about/company-info/?srslt...


In [45]:
def query_wayback_availability(url, year, max_retries=5):
    """
    Query Wayback Availability API for the closest archived snapshot
    around July 1 of a given year.
    """
    api_url = "https://archive.org/wayback/available"
    target_timestamp = f"{year}0701"

    params = {
        "url": url,
        "timestamp": target_timestamp
    }

    for attempt in range(max_retries):
        try:
            response = requests.get(
                api_url,
                params=params,
                timeout=30,
                headers={"User-Agent": "Mozilla/5.0"}
            )

            if response.status_code != 200:
                print(f"Attempt {attempt + 1}: failed for {url} {year}: {response.status_code}")
                time.sleep(5 * (attempt + 1))
                continue

            data = response.json()

            closest = data.get("archived_snapshots", {}).get("closest")

            if closest is None:
                return None

            if closest.get("available") is not True:
                return None

            snapshot_timestamp = closest.get("timestamp")
            snapshot_url = closest.get("url")
            status = closest.get("status")

            return {
                "snapshot_timestamp": snapshot_timestamp,
                "snapshot_url": snapshot_url,
                "statuscode": status,
                "target_timestamp": target_timestamp
            }

        except Exception as e:
            print(f"Attempt {attempt + 1}: error for {url} {year}: {e}")
            time.sleep(5 * (attempt + 1))

    return None

In [46]:
test_company = companies.iloc[0]
test_url = test_company["about_url"]

print(test_company["ticker"])
print(test_url)

selected = query_wayback_availability(test_url, 2016)
selected

MSFT
https://www.microsoft.com/en-us/about


{'snapshot_timestamp': '20160702232755',
 'snapshot_url': 'http://web.archive.org/web/20160702232755/http://www.microsoft.com:80/en-us/about',
 'statuscode': '200',
 'target_timestamp': '20160701'}

In [47]:
test_results = []

for year in YEARS:
    selected = query_wayback_availability(test_url, year)

    if selected is None:
        test_results.append({
            "year": year,
            "status": "missing",
            "snapshot_timestamp": None,
            "snapshot_url": None,
            "target_timestamp": f"{year}0701",
            "statuscode": None
        })
    else:
        test_results.append({
            "year": year,
            "status": "found",
            "snapshot_timestamp": selected["snapshot_timestamp"],
            "snapshot_url": selected["snapshot_url"],
            "target_timestamp": selected["target_timestamp"],
            "statuscode": selected["statuscode"]
        })

    time.sleep(1)

pd.DataFrame(test_results)

,year,status,snapshot_timestamp,snapshot_url,target_timestamp,statuscode
0,2016,found,20160702232755,http://web.archive.org/web/20160702232755/http...,20160701,200
1,2017,found,20170701073152,http://web.archive.org/web/20170701073152/http...,20170701,200
2,2018,found,20180703085306,http://web.archive.org/web/20180703085306/http...,20180701,200
3,2019,found,20190702051350,http://web.archive.org/web/20190702051350/http...,20190701,200
4,2020,missing,None,None,20200701,None
5,2021,missing,None,None,20210701,None
6,2022,missing,None,None,20220701,None
7,2023,missing,None,None,20230701,None
8,2024,found,20240701230101,http://web.archive.org/web/20240701230101/http...,20240701,200


In [51]:
all_results = []

for idx, company in companies.iterrows():
    ticker = company["ticker"]
    company_name = company["company_name"]
    sector = company["sector"]
    about_url = company["about_url"]

    print(f"\n[{idx + 1}/{len(companies)}] Company: {ticker} - {company_name}")

    for year in YEARS:
        print(f"  Querying {year}...")

        selected = query_wayback_availability(about_url, year)

        if selected is None:
            all_results.append({
                "ticker": ticker,
                "company_name": company_name,
                "sector": sector,
                "year": year,
                "about_url": about_url,
                "snapshot_timestamp": None,
                "snapshot_url": None,
                "target_timestamp": f"{year}0701",
                "status": "missing",
                "missing_reason": "No closest snapshot found from Wayback Availability API",
                "statuscode": None
            })
        else:
            all_results.append({
                "ticker": ticker,
                "company_name": company_name,
                "sector": sector,
                "year": year,
                "about_url": about_url,
                "snapshot_timestamp": selected["snapshot_timestamp"],
                "snapshot_url": selected["snapshot_url"],
                "target_timestamp": selected["target_timestamp"],
                "status": "found",
                "missing_reason": None,
                "statuscode": selected["statuscode"]
            })

        time.sleep(1)

    # Save progress after each company
    snapshots_metadata = pd.DataFrame(all_results)
    snapshots_metadata.to_csv(OUTPUT_PATH, index=False)
    print(f"  Progress saved to {OUTPUT_PATH}")

snapshots_metadata = pd.DataFrame(all_results)
snapshots_metadata.head()


[1/50] Company: MSFT - Microsoft
  Querying 2016...
  Querying 2017...
  Querying 2018...
  Querying 2019...
  Querying 2020...
  Querying 2021...
  Querying 2022...
  Querying 2023...
  Querying 2024...
  Progress saved to /Users/sc102299/Desktop/ra/src/query_wayback.py

[2/50] Company: AAPL - Apple
  Querying 2016...
  Querying 2017...
  Querying 2018...
  Querying 2019...
  Querying 2020...
  Querying 2021...
  Querying 2022...
  Querying 2023...
  Querying 2024...
  Progress saved to /Users/sc102299/Desktop/ra/src/query_wayback.py

[3/50] Company: NVDA - NVIDIA
  Querying 2016...
  Querying 2017...
  Querying 2018...
  Querying 2019...
  Querying 2020...
  Querying 2021...
  Querying 2022...
  Querying 2023...
  Querying 2024...
  Progress saved to /Users/sc102299/Desktop/ra/src/query_wayback.py

[4/50] Company: GOOGL - Alphabet
  Querying 2016...
  Querying 2017...
  Querying 2018...
  Querying 2019...
  Querying 2020...
  Querying 2021...
  Querying 2022...
  Querying 2023...
  

,ticker,company_name,sector,year,about_url,snapshot_timestamp,snapshot_url,target_timestamp,status,missing_reason,statuscode
0,MSFT,Microsoft,Technology,2016,https://www.microsoft.com/en-us/about,20160702232755,http://web.archive.org/web/20160702232755/http...,20160701,found,None,200
1,MSFT,Microsoft,Technology,2017,https://www.microsoft.com/en-us/about,None,None,20170701,missing,No closest snapshot found from Wayback Availab...,None
2,MSFT,Microsoft,Technology,2018,https://www.microsoft.com/en-us/about,None,None,20180701,missing,No closest snapshot found from Wayback Availab...,None
3,MSFT,Microsoft,Technology,2019,https://www.microsoft.com/en-us/about,20190702051350,http://web.archive.org/web/20190702051350/http...,20190701,found,None,200
4,MSFT,Microsoft,Technology,2020,https://www.microsoft.com/en-us/about,20200701214157,http://web.archive.org/web/20200701214157/http...,20200701,found,None,200


In [3]:
snapshots_metadata = pd.read_csv("../data/snapshots_metadata.csv")

print("Shape:", snapshots_metadata.shape)
print("Companies:", snapshots_metadata["ticker"].nunique())
print("Years:", snapshots_metadata["year"].min(), "-", snapshots_metadata["year"].max())
snapshots_metadata["status"].value_counts()

Shape: (450, 11)
Companies: 50
Years: 2016 - 2024


status
found      242
missing    208
Name: count, dtype: int64

In [7]:
coverage_by_company = (
    snapshots_metadata
    .groupby(["ticker", "company_name", "sector"])["status"]
    .apply(lambda x: (x == "found").sum())
    .reset_index(name="years_found")
    .sort_values("years_found")
)

## Text Extraction and Theme Analysis

In [9]:
import re
from pathlib import Path
from bs4 import BeautifulSoup
from difflib import SequenceMatcher

In [10]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

SNAPSHOTS_PATH = DATA_DIR / "snapshots_metadata.csv"
PART1_TEXT_PATH = DATA_DIR / "part1_text_extracted.csv"
PART1_FINAL_PATH = DATA_DIR / "part1_company_year_values.csv"
PART1_PROGRESS_PATH = DATA_DIR / "part1_text_extracted_progress.csv"

In [13]:
def download_snapshot_html(snapshot_url, max_retries=3):
    """
    Download HTML from a Wayback Machine snapshot URL.
    """
    if pd.isna(snapshot_url) or snapshot_url is None:
        return None
    
    for attempt in range(max_retries):
        try:
            response = requests.get(
                snapshot_url,
                timeout=60,
                headers={"User-Agent": "Mozilla/5.0"}
            )
            
            if response.status_code == 200:
                return response.text
            
            print(f"Attempt {attempt + 1}: failed {response.status_code} for {snapshot_url}")
            time.sleep(5 * (attempt + 1))
            
        except Exception as e:
            print(f"Attempt {attempt + 1}: error downloading {snapshot_url}: {e}")
            time.sleep(5 * (attempt + 1))
    
    return None

In [14]:
def clean_html_text(html):
    """
    Extract visible text from archived HTML and remove common boilerplate.
    """
    if html is None:
        return None
    
    soup = BeautifulSoup(html, "html.parser")
    
    # Remove non-content / repeated layout elements
    for tag in soup([
        "script", "style", "noscript", "svg",
        "header", "footer", "nav", "form"
    ]):
        tag.decompose()
    
    text = soup.get_text(separator=" ")
    
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    # Remove common Wayback / webpage boilerplate
    boilerplate_phrases = [
        "Wayback Machine",
        "Internet Archive",
        "Save Page Now",
        "capture a web page as it appears now",
        "Share on Facebook",
        "Share on Twitter",
        "Skip to main content",
        "Privacy Policy",
        "Terms of Use",
        "Cookie Policy"
    ]
    
    for phrase in boilerplate_phrases:
        text = text.replace(phrase, " ")
    
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [15]:
sample_row = snapshots_metadata[snapshots_metadata["status"] == "found"].iloc[0]

print(sample_row["ticker"], sample_row["year"])
print(sample_row["snapshot_url"])

sample_html = download_snapshot_html(sample_row["snapshot_url"])
sample_text = clean_html_text(sample_html)

print("Text length:", len(sample_text) if sample_text else 0)
print(sample_text[:1000] if sample_text else "No text extracted")

MSFT 2016
http://web.archive.org/web/20160702232755/http://www.microsoft.com:80/en-us/about
Text length: 3656
About Microsoft Store Store home Devices Microsoft Surface PCs & tablets Xbox Virtual reality Accessories Windows phone Microsoft Band Software Office Windows Additional software Apps All apps Windows apps Windows phone apps Games Xbox One games Xbox 360 games PC games Windows games Windows phone games Entertainment All Entertainment Movies & TV Music Business & Education Business Store Education Store Developer Sale Sale Find a store Gift cards Products Software & services Windows Office Free downloads & security Internet Explorer Microsoft Edge Skype OneNote OneDrive Microsoft Health MSN Bing Microsoft Groove Microsoft Movies & TV Devices & Xbox All Microsoft devices Microsoft Surface All Windows PCs & tablets PC accessories Xbox & games Microsoft Band Microsoft Lumia All Windows phones Microsoft HoloLens For business Cloud Platform Microsoft Azure Microsoft Dynamics Windows 

In [17]:
processed_rows = []

for idx, row in snapshots_metadata.iterrows():
    ticker = row["ticker"]
    year = row["year"]
    status = row["status"]
    snapshot_url = row["snapshot_url"]
    
    print(f"[{idx + 1}/{len(snapshots_metadata)}] {ticker} {year}")
    
    row_dict = row.to_dict()
    
    if status != "found" or pd.isna(snapshot_url):
        row_dict["page_text_clean"] = None
        row_dict["text_length"] = 0
        row_dict["text_extraction_status"] = "missing_snapshot"
    else:
        html = download_snapshot_html(snapshot_url)
        clean_text = clean_html_text(html)
        
        if clean_text is None or len(clean_text) < 100:
            row_dict["page_text_clean"] = clean_text
            row_dict["text_length"] = 0 if clean_text is None else len(clean_text)
            row_dict["text_extraction_status"] = "failed_or_too_short"
        else:
            row_dict["page_text_clean"] = clean_text
            row_dict["text_length"] = len(clean_text)
            row_dict["text_extraction_status"] = "success"
    
    processed_rows.append(row_dict)
    
    # Save progress every 25 rows
    if (idx + 1) % 25 == 0:
        temp_df = pd.DataFrame(processed_rows)
        temp_df.to_csv(PART1_PROGRESS_PATH, index=False)
        print(f"Progress saved to {PART1_PROGRESS_PATH}")
    
    time.sleep(1)

text_df = pd.DataFrame(processed_rows)
text_df.head()

[1/450] MSFT 2016
[2/450] MSFT 2017
[3/450] MSFT 2018
[4/450] MSFT 2019
[5/450] MSFT 2020
[6/450] MSFT 2021
[7/450] MSFT 2022
[8/450] MSFT 2023
[9/450] MSFT 2024
[10/450] AAPL 2016
[11/450] AAPL 2017
[12/450] AAPL 2018
[13/450] AAPL 2019
[14/450] AAPL 2020
[15/450] AAPL 2021
[16/450] AAPL 2022
[17/450] AAPL 2023
[18/450] AAPL 2024
[19/450] NVDA 2016
[20/450] NVDA 2017
[21/450] NVDA 2018
[22/450] NVDA 2019
[23/450] NVDA 2020
[24/450] NVDA 2021
[25/450] NVDA 2022
Progress saved to ../data/part1_text_extracted_progress.csv
[26/450] NVDA 2023
[27/450] NVDA 2024
Attempt 1: error downloading http://web.archive.org/web/20240701211926/https://www.nvidia.com/en-us/about-nvidia/: HTTPConnectionPool(host='web.archive.org', port=80): Max retries exceeded with url: /web/20240701211926/https://www.nvidia.com/en-us/about-nvidia/ (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x148e48a00>, 'Connection to web.archive.org timed out. (connect timeout=60)'))
[28/450] GOOGL 201

,ticker,company_name,sector,year,about_url,snapshot_timestamp,snapshot_url,target_timestamp,status,missing_reason,statuscode,page_text_clean,text_length,text_extraction_status
0,MSFT,Microsoft,Technology,2016,https://www.microsoft.com/en-us/about,2.016070e+13,http://web.archive.org/web/20160702232755/http...,20160701,found,NaN,200.0,About Microsoft Store Store home Devices Micro...,3656,success
1,MSFT,Microsoft,Technology,2017,https://www.microsoft.com/en-us/about,NaN,NaN,20170701,missing,No closest snapshot found from Wayback Availab...,NaN,None,0,missing_snapshot
2,MSFT,Microsoft,Technology,2018,https://www.microsoft.com/en-us/about,NaN,NaN,20180701,missing,No closest snapshot found from Wayback Availab...,NaN,None,0,missing_snapshot
3,MSFT,Microsoft,Technology,2019,https://www.microsoft.com/en-us/about,2.019070e+13,http://web.archive.org/web/20190702051350/http...,20190701,found,NaN,200.0,About Microsoft | Mission and Vision | Microso...,1539,success
4,MSFT,Microsoft,Technology,2020,https://www.microsoft.com/en-us/about,2.020070e+13,http://web.archive.org/web/20200701214157/http...,20200701,found,NaN,200.0,About Microsoft | Mission and Vision | Microso...,1563,success


In [18]:
print(text_df.shape)
text_df["text_extraction_status"].value_counts()

(450, 14)


text_extraction_status
success                226
missing_snapshot       208
failed_or_too_short     16
Name: count, dtype: int64

In [19]:
text_df.to_csv(PART1_TEXT_PATH, index=False)

print(f"Saved text extraction output to: {PART1_TEXT_PATH}")

Saved text extraction output to: ../data/part1_text_extracted.csv


In [20]:
from difflib import SequenceMatcher

def calculate_text_similarity(text1, text2):
    """
    Calculate text similarity between two cleaned pages.
    Returns a value from 0 to 1.
    """
    if pd.isna(text1) or pd.isna(text2) or text1 is None or text2 is None:
        return None
    
    text1_short = str(text1)[:20000]
    text2_short = str(text2)[:20000]
    
    return SequenceMatcher(None, text1_short, text2_short).ratio()

In [21]:
text_df = text_df.sort_values(["ticker", "year"]).reset_index(drop=True)

final_rows = []

for ticker, group in text_df.groupby("ticker"):
    group = group.sort_values("year").copy()
    
    prev_text = None
    
    for _, row in group.iterrows():
        current_text = row["page_text_clean"]
        row_dict = row.to_dict()
        
        if pd.isna(current_text) or current_text is None or row["text_extraction_status"] != "success":
            row_dict["similarity_to_prior"] = None
            row_dict["changed_from_prior"] = "missing_current"
        
        elif prev_text is None:
            row_dict["similarity_to_prior"] = None
            row_dict["changed_from_prior"] = "no_prior_available"
            prev_text = current_text
        
        else:
            similarity = calculate_text_similarity(prev_text, current_text)
            row_dict["similarity_to_prior"] = similarity
            
            if similarity is None:
                row_dict["changed_from_prior"] = "unknown"
            elif similarity < 0.90:
                row_dict["changed_from_prior"] = "changed"
            else:
                row_dict["changed_from_prior"] = "not_changed"
            
            prev_text = current_text
        
        final_rows.append(row_dict)

part1_df = pd.DataFrame(final_rows)

part1_df[[
    "ticker", "year", "text_extraction_status",
    "similarity_to_prior", "changed_from_prior"
]].head(30)

,ticker,year,text_extraction_status,similarity_to_prior,changed_from_prior
0,AAPL,2016,failed_or_too_short,NaN,missing_current
1,AAPL,2017,missing_snapshot,NaN,missing_current
2,AAPL,2018,failed_or_too_short,NaN,missing_current
3,AAPL,2019,missing_snapshot,NaN,missing_current
4,AAPL,2020,failed_or_too_short,NaN,missing_current
5,AAPL,2021,failed_or_too_short,NaN,missing_current
6,AAPL,2022,failed_or_too_short,NaN,missing_current
7,AAPL,2023,missing_snapshot,NaN,missing_current
8,AAPL,2024,missing_snapshot,NaN,missing_current
9,ABBV,2016,success,NaN,no_prior_available


In [22]:
part1_df["changed_from_prior"].value_counts()

changed_from_prior
missing_current       224
not_changed           137
changed                45
no_prior_available     44
Name: count, dtype: int64

In [23]:
VALUE_THEME_KEYWORDS = {
    "innovation": [
        "innovation", "innovative", "technology", "research", "development",
        "digital", "ai", "artificial intelligence", "science", "engineering"
    ],
    "customer_focus": [
        "customer", "customers", "client", "clients", "consumer", "users",
        "service", "experience", "satisfaction"
    ],
    "trust_ethics": [
        "trust", "ethics", "ethical", "integrity", "responsibility",
        "responsible", "transparency", "accountability", "compliance"
    ],
    "people_employees": [
        "people", "employees", "employee", "workforce", "talent",
        "team", "culture", "workplace", "leadership"
    ],
    "diversity_inclusion": [
        "diversity", "inclusion", "inclusive", "equity", "belonging",
        "gender", "racial", "ethnic", "underrepresented"
    ],
    "sustainability_social_impact": [
        "sustainability", "sustainable", "climate", "environment",
        "community", "social impact", "carbon", "emissions", "planet"
    ],
    "performance_growth": [
        "growth", "performance", "value", "shareholder", "profit",
        "market", "scale", "global", "leader", "leadership"
    ],
    "safety_security": [
        "safety", "security", "privacy", "secure", "protect",
        "risk", "data security", "cybersecurity"
    ]
}

In [24]:
def classify_value_themes(text, keyword_dict, min_count=2):
    """
    Classify value themes based on keyword mentions.
    """
    if text is None or pd.isna(text):
        return []
    
    text_lower = str(text).lower()
    theme_counts = {}
    
    for theme, keywords in keyword_dict.items():
        count = 0
        for kw in keywords:
            count += text_lower.count(kw.lower())
        theme_counts[theme] = count
    
    selected_themes = [
        theme for theme, count in theme_counts.items()
        if count >= min_count
    ]
    
    return selected_themes

In [25]:
part1_df["theme_categories"] = part1_df["page_text_clean"].apply(
    lambda text: classify_value_themes(text, VALUE_THEME_KEYWORDS, min_count=2)
)

In [29]:
def generate_analyst_notes(row):
    """
    Generate short analyst notes based on text extraction, themes, and change status.
    """
    if row["text_extraction_status"] != "success":
        return "No usable page text was extracted for this company-year."
    
    themes = row["theme_categories"]
    changed = row["changed_from_prior"]
    text_length = row["text_length"]
    
    if isinstance(themes, list):
        theme_text = ", ".join(themes) if themes else "no strong theme identified"
    else:
        theme_text = str(themes)
    
    if changed == "changed":
        change_note = "The page appears to have changed compared with the prior available year."
    elif changed == "not_changed":
        change_note = "The page appears similar to the prior available year."
    elif changed == "no_prior_available":
        change_note = "This is the first available page for this company in the dataset."
    else:
        change_note = "Change from the prior year could not be evaluated."
    
    return (
        f"The extracted page text is {text_length} characters long. "
        f"Main identified themes: {theme_text}. "
        f"{change_note}"
    )
part1_df["analyst_notes"] = part1_df.apply(generate_analyst_notes, axis=1)
part1_df["theme_categories"] = part1_df["theme_categories"].apply(
    lambda x: "; ".join(x) if isinstance(x, list) else x
)

In [30]:
PART1_FINAL_PATH = DATA_DIR / "part1_company_year_values.csv"

required_cols = [
    "ticker",
    "company_name",
    "sector",
    "year",
    "page_text_clean",
    "changed_from_prior",
    "theme_categories",
    "analyst_notes"
]

extra_cols = [
    "about_url",
    "snapshot_url",
    "snapshot_timestamp",
    "status",
    "text_extraction_status",
    "text_length",
    "similarity_to_prior"
]

final_cols = required_cols + [col for col in extra_cols if col in part1_df.columns]

part1_final = part1_df[final_cols].copy()

part1_final.to_csv(PART1_FINAL_PATH, index=False)

print(f"Saved final Part 1 output to: {PART1_FINAL_PATH}")
print(part1_final.shape)

Saved final Part 1 output to: ../data/part1_company_year_values.csv
(450, 15)


In [31]:
print("Shape:", part1_final.shape)
print("Companies:", part1_final["ticker"].nunique())
print("Years:", part1_final["year"].min(), "-", part1_final["year"].max())

print("\nChanged from prior:")
print(part1_final["changed_from_prior"].value_counts())

print("\nTheme counts:")
theme_counts = (
    part1_final["theme_categories"]
    .dropna()
    .str.get_dummies(sep="; ")
    .sum()
    .sort_values(ascending=False)
)

theme_counts

Shape: (450, 15)
Companies: 50
Years: 2016 - 2024

Changed from prior:
changed_from_prior
missing_current       224
not_changed           137
changed                45
no_prior_available     44
Name: count, dtype: int64

Theme counts:


performance_growth              205
people_employees                201
innovation                      192
customer_focus                  167
trust_ethics                    137
sustainability_social_impact    126
safety_security                  98
diversity_inclusion              75
dtype: int64

In [33]:
part1_final[["ticker", "year", "status", "text_extraction_status", "text_length", "changed_from_prior", "theme_categories"]].head(20)

,ticker,year,status,text_extraction_status,text_length,changed_from_prior,theme_categories
0,AAPL,2016,found,failed_or_too_short,18,missing_current,
1,AAPL,2017,missing,missing_snapshot,0,missing_current,
2,AAPL,2018,found,failed_or_too_short,18,missing_current,
3,AAPL,2019,missing,missing_snapshot,0,missing_current,
4,AAPL,2020,found,failed_or_too_short,18,missing_current,
5,AAPL,2021,found,failed_or_too_short,18,missing_current,
6,AAPL,2022,found,failed_or_too_short,18,missing_current,
7,AAPL,2023,missing,missing_snapshot,0,missing_current,
8,AAPL,2024,missing,missing_snapshot,0,missing_current,
9,ABBV,2016,found,success,3831,no_prior_available,innovation; customer_focus; trust_ethics; peop...
